In [1]:
# ── CELL 1 — IMPORTS ─────────────────────────────────────────────
import os
import pickle
import numpy as np
import pandas as pd
import torch
import datasets
import transformers
from collections import Counter
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification

print('pandas       :', pd.__version__)
print('torch        :', torch.__version__)
print('transformers :', transformers.__version__)
print('datasets     :', datasets.__version__)
print()
print('All imports successful!')

c:\Users\hp\Desktop\med\venv_med\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pandas       : 2.3.3
torch        : 2.12.0+cpu
transformers : 4.57.6
datasets     : 2.18.0

All imports successful!


In [2]:
# ── CELL 2 — LOAD DATASET ────────────────────────────────────────
dataset = load_dataset('bigbio/bc5cdr', trust_remote_code=True)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['passages'],
        num_rows: 500
    })
    test: Dataset({
        features: ['passages'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['passages'],
        num_rows: 500
    })
})


In [3]:
# ── CELL 3 — LOAD TOKENIZER ──────────────────────────────────────
# local_files_only=True uses the cached download — no internet needed
tokenizer = AutoTokenizer.from_pretrained(
    'dmis-lab/biobert-base-cased-v1.2',
    local_files_only=True
)
print('Tokenizer loaded!')
print('Vocabulary size:', tokenizer.vocab_size)

Tokenizer loaded!
Vocabulary size: 28996


In [25]:
import pickle
import torch

MAX_LEN = 512

with open(r'C:\Users\hp\Desktop\med\preprocessed_bc5cdr_all_splits.pkl', 'rb') as f:
    data = pickle.load(f)

all_input_ids = []
all_masks = []
all_labels = []

for split in ['train', 'validation', 'test']:
    tokens_list = data[split]['tokens']
    labels_list = data[split]['labels']
    
    for tokens, labels in zip(tokens_list, labels_list):
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        
        # Pad or truncate to MAX_LEN
        padding_len = MAX_LEN - len(input_ids)
        attention_mask = [1] * len(input_ids) + [0] * padding_len
        input_ids      = input_ids + [0] * padding_len
        labels         = labels    + [-100] * padding_len
        
        all_input_ids.append(input_ids)
        all_masks.append(attention_mask)
        all_labels.append(labels)

print(f"Total passages: {len(all_input_ids)}")  # should be 3000

save_data = {
    'input_ids':       all_input_ids,
    'attention_masks': all_masks,
    'labels':          all_labels
}

with open(r'C:\Users\hp\Desktop\med\colab_training_data_v2.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print("Saved colab_training_data_v2.pkl ✓")

Total passages: 3000
Saved colab_training_data_v2.pkl ✓


In [4]:
# ── CELL 4 — PREPROCESSING FUNCTION ─────────────────────────────
# Converts raw text + entity offsets → tokens + BIO labels

def preprocess_passage(text, entities, tokenizer):
    label2id = {
        'O':          0,
        'B-Chemical': 1,
        'I-Chemical': 2,
        'B-Disease':  3,
        'I-Disease':  4,
    }

    # One label per character, default O
    char_labels = ['O'] * len(text)

    for entity in entities:
        for start, end in entity['offsets']:
            # Skip broken offsets that go beyond text length
            if start >= len(text) or end > len(text):
                continue
            entity_type = entity['type']
            char_labels[start] = f'B-{entity_type}'
            for i in range(start + 1, end):
                char_labels[i] = f'I-{entity_type}'

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False,
        truncation=True,
        max_length=512
    )

    tokens        = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
    offset_mapping = encoding['offset_mapping']
    token_labels  = []

    for token, (char_start, char_end) in zip(tokens, offset_mapping):
        if char_start == char_end:       # special token
            token_labels.append(-100)
            continue
        if token.startswith('##'):       # subword continuation
            token_labels.append(-100)
            continue
        label_str = char_labels[char_start]
        token_labels.append(label2id[label_str])

    return tokens, token_labels

print('Preprocessing function defined!')

Preprocessing function defined!


In [5]:
# ── CELL 5 — PROCESS ALL THREE SPLITS ───────────────────────────
# Using BC5CDR's own train/validation/test splits correctly

all_data = {}  # stores tokens and labels for each split

for split_name in ['train', 'validation', 'test']:
    split_tokens = []
    split_labels = []
    skipped      = 0

    for row in dataset[split_name]:
        for passage in row['passages']:
            text     = passage['text']
            entities = passage['entities']

            if not text.strip():
                skipped += 1
                continue

            tokens, labels = preprocess_passage(text, entities, tokenizer)

            if len(tokens) > 0:
                split_tokens.append(tokens)
                split_labels.append(labels)

    all_data[split_name] = {
        'tokens': split_tokens,
        'labels': split_labels
    }
    print(f'{split_name:12} → {len(split_tokens)} passages processed, {skipped} skipped')

print()
print('All 3 splits processed!')

train        → 1000 passages processed, 0 skipped
validation   → 1000 passages processed, 0 skipped
test         → 1000 passages processed, 0 skipped

All 3 splits processed!


In [6]:
# ── CELL 6 — SAVE PREPROCESSED DATA ─────────────────────────────
save_dir  = r'C:\Users\hp\Desktop\snomed_ner'
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, 'preprocessed_bc5cdr_all_splits.pkl')

with open(save_path, 'wb') as f:
    pickle.dump(all_data, f)

# Verify immediately
with open(save_path, 'rb') as f:
    verify = pickle.load(f)

print('Saved and verified!')
print(f'File size : {os.path.getsize(save_path)/(1024*1024):.1f} MB')
for split_name in ['train', 'validation', 'test']:
    print(f'  {split_name:12}: {len(verify[split_name]["tokens"])} passages')

Saved and verified!
File size : 4.4 MB
  train       : 1000 passages
  validation  : 1000 passages
  test        : 1000 passages


In [7]:
# ── CELL 7 — CONVERT TO INPUT IDS + ADD CLS/SEP ─────────────────
# Convert token strings → integer IDs and add special tokens

CLS_ID = tokenizer.cls_token_id
SEP_ID = tokenizer.sep_token_id
PAD_ID = tokenizer.pad_token_id
MAX_LEN = 512

print(f'CLS={CLS_ID}  SEP={SEP_ID}  PAD={PAD_ID}  MAX_LEN={MAX_LEN}')
print()

processed = {}  # will hold input_ids, masks, labels per split

for split_name in ['train', 'validation', 'test']:
    tokens_list = all_data[split_name]['tokens']
    labels_list = all_data[split_name]['labels']

    input_ids_list = []
    masks_list     = []
    label_ids_list = []

    for tokens, labels in zip(tokens_list, labels_list):
        # Convert tokens → integer IDs
        ids = tokenizer.convert_tokens_to_ids(tokens)

        # Add CLS at start, SEP at end, truncate to 510 to leave room
        ids    = [CLS_ID] + ids[:510] + [SEP_ID]
        lbls   = [-100]   + labels[:510] + [-100]

        # Pad to MAX_LEN
        pad_len = MAX_LEN - len(ids)
        mask    = [1] * len(ids) + [0] * pad_len
        ids     = ids  + [PAD_ID] * pad_len
        lbls    = lbls + [-100]   * pad_len

        input_ids_list.append(ids)
        masks_list.append(mask)
        label_ids_list.append(lbls)

    processed[split_name] = {
        'input_ids': input_ids_list,
        'masks':     masks_list,
        'labels':    label_ids_list
    }
    print(f'{split_name:12} → {len(input_ids_list)} sequences ready')

print()
# Quick shape check
sample = processed['train']['input_ids'][0]
print(f'Sample sequence length: {len(sample)} (should be {MAX_LEN})')
print('All same length?', len(set(len(x) for x in processed['train']['input_ids'])) == 1)

CLS=101  SEP=102  PAD=0  MAX_LEN=512

train        → 1000 sequences ready
validation   → 1000 sequences ready
test         → 1000 sequences ready

Sample sequence length: 512 (should be 512)
All same length? True


In [8]:
# ── CELL 8 — BUILD PYTORCH DATASETS AND DATALOADERS ─────────────

class NERDataset(Dataset):
    def __init__(self, input_ids, attention_masks, label_ids):
        self.input_ids       = torch.tensor(input_ids,       dtype=torch.long)
        self.attention_masks = torch.tensor(attention_masks, dtype=torch.long)
        self.label_ids       = torch.tensor(label_ids,       dtype=torch.long)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels':         self.label_ids[idx]
        }

train_dataset = NERDataset(
    processed['train']['input_ids'],
    processed['train']['masks'],
    processed['train']['labels']
)
val_dataset = NERDataset(
    processed['validation']['input_ids'],
    processed['validation']['masks'],
    processed['validation']['labels']
)
test_dataset = NERDataset(
    processed['test']['input_ids'],
    processed['test']['masks'],
    processed['test']['labels']
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False)

print(f'Train   : {len(train_dataset)} sequences, {len(train_loader)} batches')
print(f'Val     : {len(val_dataset)} sequences, {len(val_loader)} batches')
print(f'Test    : {len(test_dataset)} sequences, {len(test_loader)} batches')
print()

# Shape check
batch = next(iter(train_loader))
print('Batch shapes:')
print('  input_ids     :', batch['input_ids'].shape)
print('  attention_mask:', batch['attention_mask'].shape)
print('  labels        :', batch['labels'].shape)

Train   : 1000 sequences, 125 batches
Val     : 1000 sequences, 125 batches
Test    : 1000 sequences, 125 batches

Batch shapes:
  input_ids     : torch.Size([8, 512])
  attention_mask: torch.Size([8, 512])
  labels        : torch.Size([8, 512])


In [9]:
# ── CELL 9 — SAVE COLAB TRAINING DATA ───────────────────────────
# Save the processed train/val data to upload to Colab for GPU training

colab_path = os.path.join(r'C:\Users\hp\Desktop\med', 'colab_training_data.pkl')

colab_data = {
    'train_input_ids' : processed['train']['input_ids'],
    'train_masks'     : processed['train']['masks'],
    'train_labels'    : processed['train']['labels'],
    'val_input_ids'   : processed['validation']['input_ids'],
    'val_masks'       : processed['validation']['masks'],
    'val_labels'      : processed['validation']['labels'],
}

with open(colab_path, 'wb') as f:
    pickle.dump(colab_data, f)

size_mb = os.path.getsize(colab_path) / (1024*1024)
print(f'Saved to  : {colab_path}')
print(f'File size : {size_mb:.1f} MB')
print('Ready to upload to Colab!')

Saved to  : C:\Users\hp\Desktop\med\colab_training_data.pkl
File size : 8.4 MB
Ready to upload to Colab!


In [10]:
# ── CELL 10 — EXTRACT MESH DICTIONARY ───────────────────────────
# Build MESH ID → concept name mapping from BC5CDR annotations

mesh_dict = {}  # { 'D006973': 'hypertensive' }

for split_name in ['train', 'validation', 'test']:
    for row in dataset[split_name]:
        for passage in row['passages']:
            for entity in passage['entities']:
                for norm in entity['normalized']:
                    mesh_id   = norm['db_id']
                    mesh_name = entity['text'][0]
                    if mesh_id and mesh_id not in mesh_dict:
                        mesh_dict[mesh_id] = mesh_name

print(f'Total unique MESH concepts: {len(mesh_dict)}')
print()
print('Sample entries:')
for k, v in list(mesh_dict.items())[:10]:
    print(f'  {k}  →  {v}')

Total unique MESH concepts: 2351

Sample entries:
  D009270  →  Naloxone
  D003000  →  clonidine
  D006973  →  hypertensive
  D007022  →  hypotensive
  D008750  →  alpha-methyldopa
  D008012  →  Lidocaine
  D006323  →  cardiac asystole
  D003866  →  depression
  D001919  →  bradyarrhythmias
  D013390  →  Suxamethonium


In [11]:
# ── CELL 11 — SAVE MESH DICTIONARY ──────────────────────────────
mesh_path = os.path.join(r'C:\Users\hp\Desktop\snomed_ner', 'mesh_dict.pkl')

with open(mesh_path, 'wb') as f:
    pickle.dump(mesh_dict, f)

print(f'MESH dictionary saved to: {mesh_path}')
print(f'Total concepts saved: {len(mesh_dict)}')

MESH dictionary saved to: C:\Users\hp\Desktop\snomed_ner\mesh_dict.pkl
Total concepts saved: 2351


In [12]:
# ── CELL 12 — BUILD MESH LINKER ──────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np
import pickle
import os

# Load the sentence transformer model
# This model converts text → meaning vectors (embeddings)
# all-MiniLM-L6-v2 is small, fast, and works well for medical text
print("Loading sentence transformer model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")
print()

# Load our saved MESH dictionary
mesh_path = r'C:\Users\hp\Desktop\snomed_ner\mesh_dict.pkl'
with open(mesh_path, 'rb') as f:
    mesh_dict = pickle.load(f)

print(f"MESH concepts loaded: {len(mesh_dict)}")
print()

# Convert all MESH concept names → embedding vectors
# We do this ONCE and save it — so at inference time
# we don't recompute 2351 embeddings every single time

print("Building MESH embeddings (this takes ~30 seconds)...")

mesh_ids   = list(mesh_dict.keys())    # ['D009270', 'D003000', ...]
mesh_names = list(mesh_dict.values())  # ['Naloxone', 'clonidine', ...]

# encode() converts each name into a 384-dimensional vector
# batch_size=64 means process 64 names at once
# show_progress_bar=True shows a progress bar
mesh_embeddings = embedder.encode(
    mesh_names,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print()
print(f"Embeddings shape: {mesh_embeddings.shape}")
print(f"  rows    = {mesh_embeddings.shape[0]} (one per MESH concept)")
print(f"  columns = {mesh_embeddings.shape[1]} (dimensions per vector)")

Loading sentence transformer model...
Model loaded!

MESH concepts loaded: 2351

Building MESH embeddings (this takes ~30 seconds)...


Batches: 100%|██████████| 37/37 [01:32<00:00,  2.51s/it]


Embeddings shape: (2351, 384)
  rows    = 2351 (one per MESH concept)
  columns = 384 (dimensions per vector)


In [13]:
# ── CELL 13 — SAVE EMBEDDINGS + DEFINE LINKER ────────────────────

# Save embeddings so we never recompute them
embed_save = {
    'mesh_ids'        : mesh_ids,
    'mesh_names'      : mesh_names,
    'mesh_embeddings' : mesh_embeddings
}

embed_path = r'C:\Users\hp\Desktop\snomed_ner\mesh_embeddings.pkl'
with open(embed_path, 'wb') as f:
    pickle.dump(embed_save, f)

size_mb = os.path.getsize(embed_path) / (1024*1024)
print(f"Embeddings saved! File size: {size_mb:.1f} MB")
print()

# ── THE LINKER FUNCTION ───────────────────────────────────────────
def link_to_mesh(span_text, embedder, mesh_embeddings, mesh_ids, mesh_names, top_k=3):
    """
    Given a medical span like 'hypertensive disorder',
    find the top_k most similar MESH concepts.
    
    Returns a list of (mesh_id, mesh_name, similarity_score)
    """
    # Step 1 — embed the input span
    span_embedding = embedder.encode(
        [span_text],
        convert_to_numpy=True
    )  # shape: (1, 384)

    # Step 2 — compute cosine similarity between span and all MESH concepts
    # Cosine similarity = dot product of normalised vectors
    # Result is a score between -1 and 1 (higher = more similar)
    
    # Normalise both vectors first
    span_norm = span_embedding / np.linalg.norm(span_embedding, axis=1, keepdims=True)
    mesh_norm = mesh_embeddings / np.linalg.norm(mesh_embeddings, axis=1, keepdims=True)
    
    # Dot product gives cosine similarity for each MESH concept
    similarities = np.dot(mesh_norm, span_norm.T).flatten()  # shape: (2351,)

    # Step 3 — get top_k highest similarity scores
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            'mesh_id'   : mesh_ids[idx],
            'mesh_name' : mesh_names[idx],
            'score'     : round(float(similarities[idx]), 4)
        })

    return results

print("Linker function defined!")

Embeddings saved! File size: 3.5 MB

Linker function defined!


In [14]:
# ── CELL 14 — TEST THE LINKER ─────────────────────────────────────
# Test with medical terms we know are in the MESH dictionary

test_spans = [
    "hypertensive disorder",
    "clonidine",
    "cardiac arrest",
    "depression",
    "naloxone",
    "high blood pressure",    # not exact — tests semantic matching
    "seizure",                # not in dict — tests generalisation
]

print(f"{'SPAN':<30} {'TOP MATCH':<20} {'MESH ID':<12} {'SCORE'}")
print("-" * 75)

for span in test_spans:
    results = link_to_mesh(
        span,
        embedder,
        mesh_embeddings,
        mesh_ids,
        mesh_names,
        top_k=1
    )
    top = results[0]
    print(f"{span:<30} {top['mesh_name']:<20} {top['mesh_id']:<12} {top['score']}")

print()
print("Top 3 matches for 'high blood pressure':")
results = link_to_mesh("high blood pressure", embedder, mesh_embeddings, mesh_ids, mesh_names, top_k=3)
for r in results:
    print(f"  {r['mesh_name']:<25} {r['mesh_id']:<12} score: {r['score']}")

SPAN                           TOP MATCH            MESH ID      SCORE
---------------------------------------------------------------------------
hypertensive disorder          hypertensive         D006973      0.8712
clonidine                      clonidine            D003000      1.0
cardiac arrest                 sudden cardiac death D016757      0.6549
depression                     depression           D003866      1.0
naloxone                       Naloxone             D009270      1.0
high blood pressure            hypertensive         D006973      0.6164
seizure                        seizures             D012640      0.9233

Top 3 matches for 'high blood pressure':
  hypertensive              D006973      score: 0.6164
  diastolic hypertension    C563897      score: 0.6034
  malignant hypertension    D006974      score: 0.5557


In [15]:
# ── CELL 15 — LOAD TRAINED BIOBERT MODEL ─────────────────────────
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch

model_path = r'C:\Users\hp\Desktop\med\biobert_ner_model'

# First verify the files are actually there
print("Files in model folder:")
for f in os.listdir(model_path):
    size = os.path.getsize(os.path.join(model_path, f)) / (1024*1024)
    print(f"  {f} — {size:.1f} MB")
print()

model = AutoModelForTokenClassification.from_pretrained(
    model_path,
    num_labels=5,
    ignore_mismatched_sizes=True,
    local_files_only=True    # ← this tells it to use local path
)
model.eval()

device = torch.device('cpu')
model  = model.to(device)

id2label = {
    0: 'O',
    1: 'B-Chemical',
    2: 'I-Chemical',
    3: 'B-Disease',
    4: 'I-Disease'
}

print("Trained NER model loaded!")
print(f"Device: {device}")

Files in model folder:
  config.json — 0.0 MB
  model.safetensors — 411.0 MB
  tokenizer.json — 0.6 MB
  tokenizer_config.json — 0.0 MB
  vocab.txt — 0.2 MB

Trained NER model loaded!
Device: cpu


In [16]:
# ── CELL 16 — FULL PIPELINE (FIXED) ──────────────────────────────

def run_pipeline(text, model, tokenizer, embedder,
                 mesh_embeddings, mesh_ids, mesh_names,
                 device, id2label):

    # ── STEP 1: TOKENIZE ─────────────────────────────────────────
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    offset_mapping = encoding['offset_mapping'][0].tolist()
    tokens         = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    # ── STEP 2: RUN NER MODEL ────────────────────────────────────
    with torch.no_grad():
        outputs     = model(input_ids=input_ids, attention_mask=attention_mask)
    logits      = outputs.logits[0]
    predictions = torch.argmax(logits, dim=-1).tolist()

    # ── STEP 3: RECONSTRUCT WORD-LEVEL SPANS ─────────────────────
    # The fix: when we see a ## subword token, we extend the current
    # word's end position instead of treating it as a new token.
    # We only assign a label to the FIRST subword of each word.

    word_tokens = []   # one entry per WORD (not subword)

    i = 0
    while i < len(tokens):
        token      = tokens[i]
        pred       = predictions[i]
        char_start = offset_mapping[i][0]
        char_end   = offset_mapping[i][1]

        # Skip special tokens [CLS] and [SEP]
        if char_start == char_end:
            i += 1
            continue

        # Skip ## subword continuations — they belong to previous word
        if token.startswith('##'):
            # Extend the last word's end position
            if word_tokens:
                word_tokens[-1]['char_end'] = char_end
            i += 1
            continue

        # This is a real first token of a word
        # Look ahead and merge all following ## tokens into this word
        j = i + 1
        while j < len(tokens) and tokens[j].startswith('##'):
            char_end = offset_mapping[j][1]
            j += 1

        word_tokens.append({
            'label'     : id2label[pred],
            'char_start': char_start,
            'char_end'  : char_end
        })

        i = j  # jump past all the ## tokens we already merged

    # ── STEP 4: GROUP CONSECUTIVE B/I LABELS INTO SPANS ──────────
    spans = []
    current_start = None
    current_end   = None
    current_type  = None

    for wt in word_tokens:
        label = wt['label']

        if label.startswith('B-'):
            # Save previous span
            if current_start is not None:
                spans.append({
                    'text'      : text[current_start:current_end].strip(),
                    'type'      : current_type,
                    'char_start': current_start,
                    'char_end'  : current_end
                })
            # Start new span
            current_start = wt['char_start']
            current_end   = wt['char_end']
            current_type  = label[2:]  # remove 'B-'

        elif label.startswith('I-') and current_start is not None:
            # Extend current span
            current_end = wt['char_end']

        else:
            # O label — close any open span
            if current_start is not None:
                spans.append({
                    'text'      : text[current_start:current_end].strip(),
                    'type'      : current_type,
                    'char_start': current_start,
                    'char_end'  : current_end
                })
            current_start = None
            current_end   = None
            current_type  = None

    # Save last span if text ends inside an entity
    if current_start is not None:
        spans.append({
            'text'      : text[current_start:current_end].strip(),
            'type'      : current_type,
            'char_start': current_start,
            'char_end'  : current_end
        })

    # ── STEP 5: LINK EACH SPAN TO MESH ───────────────────────────
    results = []
    for span in spans:
        if not span['text']:
            continue
        matches   = link_to_mesh(span['text'], embedder,
                                 mesh_embeddings, mesh_ids, mesh_names, top_k=1)
        top_match = matches[0]
        results.append({
            'span'      : span['text'],
            'type'      : span['type'],
            'char_start': span['char_start'],
            'char_end'  : span['char_end'],
            'mesh_id'   : top_match['mesh_id'],
            'mesh_name' : top_match['mesh_name'],
            'score'     : top_match['score']
        })

    return results

print("Pipeline function defined (fixed)!")

Pipeline function defined (fixed)!


In [17]:
# ── CELL 17 — TEST FULL PIPELINE ─────────────────────────────────

test_note = """
Patient is a 58-year-old male with a history of hypertension 
and depression. He was prescribed clonidine for blood pressure 
control. He also reported episodes of cardiac arrest last year. 
Current medications include naloxone and lidocaine.
"""

print("INPUT NOTE:")
print(test_note)
print()

results = run_pipeline(
    test_note.strip(),
    model,
    tokenizer,
    embedder,
    mesh_embeddings,
    mesh_ids,
    mesh_names,
    device,
    id2label
)

print(f"FOUND {len(results)} ENTITIES:")
print()
print(f"{'SPAN':<25} {'TYPE':<12} {'MESH ID':<12} {'MESH NAME':<25} {'SCORE'}")
print("-" * 85)
for r in results:
    print(f"{r['span']:<25} {r['type']:<12} {r['mesh_id']:<12} {r['mesh_name']:<25} {r['score']}")

INPUT NOTE:

Patient is a 58-year-old male with a history of hypertension 
and depression. He was prescribed clonidine for blood pressure 
control. He also reported episodes of cardiac arrest last year. 
Current medications include naloxone and lidocaine.


FOUND 2 ENTITIES:

SPAN                      TYPE         MESH ID      MESH NAME                 SCORE
-------------------------------------------------------------------------------------
clonidine                 Chemical     D003000      clonidine                 1.0
lidocaine                 Chemical     D008012      Lidocaine                 1.0


In [18]:
# ── CELL 18 — SAVE PIPELINE CONFIG ───────────────────────────────
# Save all paths into one config file so Flask can load everything

import json

config = {
    'model_path'      : r'C:\Users\hp\Desktop\med\biobert_ner_model',
    'mesh_embed_path' : r'C:\Users\hp\Desktop\med\mesh_embeddings.pkl',
    'mesh_dict_path'  : r'C:\Users\hp\Desktop\med\mesh_dict.pkl'
}

config_path = r'C:\Users\hp\Desktop\med\clinical_ner_app\config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("Config saved!")
print(json.dumps(config, indent=2))

Config saved!
{
  "model_path": "C:\\Users\\hp\\Desktop\\med\\biobert_ner_model",
  "mesh_embed_path": "C:\\Users\\hp\\Desktop\\med\\mesh_embeddings.pkl",
  "mesh_dict_path": "C:\\Users\\hp\\Desktop\\med\\mesh_dict.pkl"
}


In [19]:
import pickle

with open(r'C:\Users\hp\Desktop\med\preprocessed_bc5cdr_all_splits.pkl', 'rb') as f:
    data = pickle.load(f)

for split in ['train', 'validation', 'test']:
    print(f"{split}: {len(data[split]['tokens'])} passages")

train: 1000 passages
validation: 1000 passages
test: 1000 passages


In [20]:
# import pickle

# with open(r'C:\Users\hp\Desktop\med\preprocessed_bc5cdr_all_splits.pkl', 'rb') as f:
#     data = pickle.load(f)

# # Check sizes
# for split in ['train', 'validation', 'test']:
#     print(f"{split}: {len(data[split]['tokens'])} passages")

# # Combine all 3 splits into one training set
# all_input_ids = data['train']['input_ids'] + data['validation']['input_ids'] + data['test']['input_ids']
# all_masks     = data['train']['attention_masks'] + data['validation']['attention_masks'] + data['test']['attention_masks']
# all_labels    = data['train']['labels'] + data['validation']['labels'] + data['test']['labels']

# print(f"\nTotal passages for training: {len(all_input_ids)}")

# save_data = {
#     'input_ids':       all_input_ids,
#     'attention_masks': all_masks,
#     'labels':          all_labels
# }

# with open(r'C:\Users\hp\Desktop\med\colab_training_data_v2.pkl', 'wb') as f:
#     pickle.dump(save_data, f)

# print("Saved colab_training_data_v2.pkl")

In [22]:
import pickle

with open(r'C:\Users\hp\Desktop\med\preprocessed_bc5cdr_all_splits.pkl', 'rb') as f:
    data = pickle.load(f)

# Check sizes
for split in ['train', 'validation', 'test']:
    print(f"{split}: {len(data[split]['tokens'])} passages")

# Combine all 3 splits into one training set
all_input_ids = data['train']['input_ids'] + data['validation']['input_ids'] + data['test']['input_ids']
all_masks     = data['train']['attention_masks'] + data['validation']['attention_masks'] + data['test']['attention_masks']
all_labels    = data['train']['labels'] + data['validation']['labels'] + data['test']['labels']

print(f"\nTotal passages for training: {len(all_input_ids)}")

save_data = {
    'input_ids':       all_input_ids,
    'attention_masks': all_masks,
    'labels':          all_labels
}

with open(r'C:\Users\hp\Desktop\med\colab_training_data_v2.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print("Saved colab_training_data_v2.pkl")

train: 1000 passages
validation: 1000 passages
test: 1000 passages


KeyError: 'input_ids'

In [23]:
import pickle
with open(r'C:\Users\hp\Desktop\med\colab_training_data.pkl', 'rb') as f:
    d = pickle.load(f)
print(d.keys())
print(len(d[list(d.keys())[0]]))

dict_keys(['train_input_ids', 'train_masks', 'train_labels', 'val_input_ids', 'val_masks', 'val_labels'])
1000


In [24]:
import pickle
with open(r'C:\Users\hp\Desktop\med\preprocessed_bc5cdr_all_splits.pkl', 'rb') as f:
    data = pickle.load(f)

# What keys does each split have?
print("Train keys:", list(data['train'].keys()))
print("Train sample - first key length:", len(data['train'][list(data['train'].keys())[0]]))

Train keys: ['tokens', 'labels']
Train sample - first key length: 1000
